In [ ]:
# 1. 환경 설정 및 라이브러리 설치 (필요 시 실행)
!pip install transformers datasets -q

# 2. Colab 파일 업로드
from google.colab import files
uploaded = files.upload()

# 이제 train.csv, val.csv 같은 학습 데이터를 업로드하세요.
# CSV 형식은 "text,label"로 되어 있어야 합니다.


In [ ]:
# 3. 데이터 불러오기
import pandas as pd

df_train = pd.read_csv("train.csv")
df_val = pd.read_csv("val.csv")
df_train = df_train.dropna()
df_val = df_val.dropna()
df_train['label'] = df_train['label'].astype(int)
df_val['label'] = df_val['label'].astype(int)

# 4. 모델과 토크나이저 로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
# 5. Dataset 클래스 정의
from torch.utils.data import Dataset

class HateSpeechDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]["text"]
        label = int(self.data.iloc[idx]["label"])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": label
        }

train_dataset = HateSpeechDataset(df_train, tokenizer)
val_dataset = HateSpeechDataset(df_val, tokenizer)


In [ ]:
# 6. Trainer 구성 및 학습
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()


In [ ]:
# 7. 저장 및 추론 테스트
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

from transformers import pipeline
pipe = pipeline("text-classification", model="./fine_tuned_model", tokenizer="./fine_tuned_model")

example = "너 정말 쓸모없는 인간이야."
result = pipe(example)

labels = {0: "not_hate_speech", 1: "hate_speech"}
for r in result:
    label_id = int(r['label'].split('_')[-1]) if 'label' in r else -1
    print(f"예측 결과: {labels.get(label_id, 'Unknown')} ({r['score']:.2%} 확률)")
